# Geo Holdout Test: Measuring Paid Search Incrementality
### A Marketing Science Framework for Travel E-Commerce

**Author:** Olivia Pan  
**Domain:** Marketing Science | Causal Inference | Incrementality Measurement  

---

## The Business Problem

A travel e-commerce company is planning a **$15M paid search investment** across 50 U.S. markets for Q3. Before committing full budget, the marketing science team needs to answer one precise question:

> **How many incremental bookings does paid search actually drive — beyond what organic demand would have produced anyway?**

This is not a reporting question. It's a causal question. And the answer has direct budget implications.

Last-click attribution would overcount — users who were already going to book will click a paid ad on their way in, inflating apparent ROAS. The business needs a measurement approach that isolates *caused* conversions from *correlated* ones.

**The decision at stake:** Whether to scale, hold, or reallocate the $15M based on true incrementality — not attributed revenue.

---

## Why Geo Holdout?

Three methods are commonly considered for this type of question. Here's the honest tradeoff:

| Method | Strengths | Limitations | Fit Here? |
|---|---|---|---|
| **Media Mix Modeling (MMM)** | Full-funnel, cross-channel view | Slow to update, low granularity, hard to validate | ❌ Too slow for Q3 decision |
| **User-Level A/B Test** | Clean treatment assignment | Can't withhold paid search at user level — competitors fill the gap | ❌ Not feasible for search |
| **Geo Holdout (Ghost Ads)** | Captures system-level effects, operationally feasible | Requires sufficient markets, spillover risk between adjacent geos | ✅ Right tool for this problem |

**Geo holdout works here because:** paid search can be switched off at the market level (via geo targeting), giving us a clean treatment/control assignment. We measure the *difference in booking trajectories* between markets that saw ads and markets that didn't.

The statistical engine underneath is **Difference-in-Differences (DiD)** — a causal inference method that controls for pre-existing differences between markets by using their shared pre-period trend as a baseline.

---

## Setup & Dependencies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from statsmodels.formula.api import ols
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
BRAND_BLUE = '#003580'   # Travel brand primary
BRAND_ORANGE = '#FF6B35' # Holdout / treatment accent
GRAY = '#8C8C8C'

np.random.seed(42)
print('Environment ready.')

---

## Section 1: Experimental Design

### Market Selection Criteria

Good geo holdout design lives or dies on market selection. We need control markets that would have behaved like treatment markets *in the absence of the intervention*. This is the **parallel trends assumption** — the core validity condition for DiD.

**Selection criteria applied:**
- Minimum booking volume threshold (exclude markets too small for statistical power)
- Pre-period trend similarity (correlation > 0.85 between candidate pairs)
- Geographic separation (avoid spillover between adjacent DMAs)
- No major local events or anomalies in test window

**Design parameters:**
- 50 total markets → 35 treatment, 15 holdout (control)
- Pre-period: 12 weeks (establishes baseline trend)
- Test period: 8 weeks (sufficient power, manageable revenue risk)
- Treatment: paid search fully active in treatment markets, paused in holdout

In [ ]:
# ── Design Parameters ──────────────────────────────────────────
N_MARKETS      = 50
N_TREATMENT    = 35
N_CONTROL      = 15
PRE_WEEKS      = 12
TEST_WEEKS     = 8
TOTAL_WEEKS    = PRE_WEEKS + TEST_WEEKS

# True incrementality we're trying to recover (ground truth for simulation)
# In a real test, we don't know this — we're estimating it
TRUE_LIFT      = 0.18  # 18% incremental booking lift from paid search

print(f'Design summary:')
print(f'  Markets: {N_TREATMENT} treatment | {N_CONTROL} holdout')
print(f'  Timeline: {PRE_WEEKS}wk pre-period + {TEST_WEEKS}wk test')
print(f'  True lift injected (ground truth): {TRUE_LIFT:.0%}')
print(f'  Goal: recover ~{TRUE_LIFT:.0%} from DiD estimation')

---

## Section 2: Synthetic Data Generation

We simulate weekly bookings for 50 markets over 20 weeks. Each market has:
- A baseline booking volume (varies by market size)
- A shared seasonal trend (travel seasonality)
- Market-specific noise
- Treatment effect applied only to treatment markets in the test period

In [ ]:
weeks     = np.arange(1, TOTAL_WEEKS + 1)
is_test   = (weeks > PRE_WEEKS).astype(int)

# Shared seasonal component (summer travel uptick)
seasonal  = 1 + 0.03 * np.sin(2 * np.pi * weeks / 52) + 0.005 * weeks

records = []
for mkt_id in range(N_MARKETS):
    is_treatment = int(mkt_id < N_TREATMENT)
    base         = np.random.uniform(800, 3000)   # market size heterogeneity
    noise_sd     = base * 0.07                    # ~7% weekly noise

    for w_idx, week in enumerate(weeks):
        trend    = seasonal[w_idx]
        noise    = np.random.normal(0, noise_sd)
        lift     = (TRUE_LIFT * is_treatment * is_test[w_idx])
        bookings = base * trend * (1 + lift) + noise

        records.append({
            'market_id'    : f'MKT_{mkt_id:02d}',
            'week'         : week,
            'is_treatment' : is_treatment,
            'is_test'      : is_test[w_idx],
            'bookings'     : max(bookings, 0),
            'base_volume'  : base
        })

df = pd.DataFrame(records)
print(f'Dataset: {len(df):,} rows | {df.market_id.nunique()} markets | {df.week.nunique()} weeks')
df.head(8)

---

## Section 3: Pre-Period Validation — Parallel Trends Check

Before we can trust the DiD estimate, we must verify the **parallel trends assumption**: treatment and control markets were moving together *before* the test started.

If trends diverged pre-period, any post-period difference could be a continuation of that pre-existing gap — not a treatment effect. This is the most common failure mode in poorly designed geo tests.

In [ ]:
# Index bookings to Week 1 = 100 for comparability across market sizes
week1_avg = df[df.week == 1].groupby('is_treatment')['bookings'].mean()
df = df.merge(
    week1_avg.rename('week1_base').reset_index(),
    on='is_treatment'
)
df['bookings_indexed'] = df['bookings'] / df['week1_base'] * 100

# Aggregate by week + group
weekly = (
    df.groupby(['week', 'is_treatment'])['bookings_indexed']
    .mean()
    .reset_index()
)
treatment_weekly = weekly[weekly.is_treatment == 1]
control_weekly   = weekly[weekly.is_treatment == 0]
pre_treatment    = treatment_weekly[treatment_weekly.week <= PRE_WEEKS]
pre_control      = control_weekly[control_weekly.week <= PRE_WEEKS]

# Parallel trends correlation
corr, pval = stats.pearsonr(
    pre_treatment['bookings_indexed'].values,
    pre_control['bookings_indexed'].values
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Full timeline
ax = axes[0]
ax.axvspan(PRE_WEEKS + 0.5, TOTAL_WEEKS + 0.5, alpha=0.08, color=BRAND_ORANGE, label='Test period')
ax.axvline(PRE_WEEKS + 0.5, color=GRAY, linestyle='--', linewidth=1, alpha=0.7)
ax.plot(treatment_weekly.week, treatment_weekly.bookings_indexed,
        color=BRAND_BLUE, linewidth=2.5, label='Treatment markets (paid search active)')
ax.plot(control_weekly.week, control_weekly.bookings_indexed,
        color=BRAND_ORANGE, linewidth=2.5, linestyle='--', label='Holdout markets (paid search paused)')
ax.set_title('Indexed Bookings: Treatment vs. Holdout', fontsize=13, fontweight='bold')
ax.set_xlabel('Week')
ax.set_ylabel('Indexed Bookings (Week 1 = 100)')
ax.legend(fontsize=9)
ax.text(PRE_WEEKS + 1, ax.get_ylim()[0] + 2, 'TEST\nPERIOD', fontsize=8, color=GRAY)

# Right: Pre-period scatter
ax2 = axes[1]
ax2.scatter(
    pre_control['bookings_indexed'],
    pre_treatment['bookings_indexed'],
    color=BRAND_BLUE, alpha=0.7, s=60
)
m, b = np.polyfit(pre_control['bookings_indexed'], pre_treatment['bookings_indexed'], 1)
x_line = np.linspace(pre_control['bookings_indexed'].min(), pre_control['bookings_indexed'].max(), 100)
ax2.plot(x_line, m * x_line + b, color=BRAND_ORANGE, linewidth=2)
ax2.set_title(f'Pre-Period Parallel Trends Validation\nr = {corr:.3f}, p = {pval:.4f}', fontsize=13, fontweight='bold')
ax2.set_xlabel('Holdout Markets (indexed)')
ax2.set_ylabel('Treatment Markets (indexed)')

# Validation result
status = '✅ PASS' if corr > 0.85 else '⚠️ REVIEW REQUIRED'
ax2.text(0.05, 0.92, f'Parallel trends: {status}', transform=ax2.transAxes,
         fontsize=10, color='green' if corr > 0.85 else 'orange',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig('parallel_trends_validation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nPre-period correlation: {corr:.3f} (threshold: 0.85)')
print(f'Parallel trends assumption: {status}')

---

## Section 4: Difference-in-Differences Estimation

With parallel trends validated, we can run the DiD model.

**The DiD formula:**

```
Bookings = β₀ + β₁(Treatment) + β₂(Post) + β₃(Treatment × Post) + ε
```

Where:
- `β₁` = pre-existing difference between treatment and control (baseline gap)
- `β₂` = time trend shared by both groups (seasonality, macro)
- **`β₃` = the causal estimate we care about** — the incremental effect of paid search

By including both main effects, β₃ isolates the treatment effect from pre-existing differences *and* from shared time trends. That's the power of DiD.

In [ ]:
# Market-level aggregation for DiD
# Use mean weekly bookings per market per period for cleaner estimation
did_df = (
    df.groupby(['market_id', 'is_treatment', 'is_test'])['bookings']
    .mean()
    .reset_index()
    .rename(columns={'bookings': 'avg_weekly_bookings'})
)

# DiD regression
model  = ols('avg_weekly_bookings ~ is_treatment + is_test + is_treatment:is_test', data=did_df).fit()

# Extract the key coefficient
did_coef  = model.params['is_treatment:is_test']
did_se    = model.bse['is_treatment:is_test']
did_ci    = model.conf_int().loc['is_treatment:is_test']
did_pval  = model.pvalues['is_treatment:is_test']

# Baseline (control pre-period average) for lift % calculation
baseline  = did_df[(did_df.is_treatment == 0) & (did_df.is_test == 0)]['avg_weekly_bookings'].mean()
est_lift  = did_coef / baseline

print('=' * 55)
print('  DIFFERENCE-IN-DIFFERENCES RESULTS')
print('=' * 55)
print(f'  DiD coefficient (β₃):  {did_coef:>10.1f} bookings/week')
print(f'  Standard error:         {did_se:>10.1f}')
print(f'  95% CI:                [{did_ci[0]:.1f}, {did_ci[1]:.1f}]')
print(f'  p-value:                {did_pval:.4f}')
print(f'  Estimated lift:         {est_lift:.1%}')
print(f'  True lift (ground truth): {TRUE_LIFT:.1%}')
print(f'  Recovery accuracy:      {abs(est_lift - TRUE_LIFT) / TRUE_LIFT:.1%} error')
print('=' * 55)
print(model.summary().tables[1])

---

## Section 5: Visualizing the Causal Effect

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: DiD diagram ─────────────────────────────────────────
ax = axes[0]
pre  = weekly[weekly.week <= PRE_WEEKS]
post = weekly[weekly.week >  PRE_WEEKS]

pre_t  = pre[pre.is_treatment == 1]
pre_c  = pre[pre.is_treatment == 0]
post_t = post[post.is_treatment == 1]
post_c = post[post.is_treatment == 0]

ax.plot(pre_t.week,  pre_t.bookings_indexed,  color=BRAND_BLUE,   lw=2.5, label='Treatment')
ax.plot(pre_c.week,  pre_c.bookings_indexed,  color=BRAND_ORANGE, lw=2.5, linestyle='--', label='Holdout')
ax.plot(post_t.week, post_t.bookings_indexed, color=BRAND_BLUE,   lw=2.5)
ax.plot(post_c.week, post_c.bookings_indexed, color=BRAND_ORANGE, lw=2.5, linestyle='--')

# Counterfactual (what treatment would look like without ads)
last_pre_t = pre_t[pre_t.week == PRE_WEEKS]['bookings_indexed'].values[0]
last_pre_c = pre_c[pre_c.week == PRE_WEEKS]['bookings_indexed'].values[0]
cf_offset  = last_pre_t - last_pre_c
counterfactual = post_c.bookings_indexed + cf_offset
ax.plot(post_c.week, counterfactual, color=BRAND_BLUE, lw=1.5, linestyle=':', alpha=0.6, label='Counterfactual')

# Shade the incremental gap
ax.fill_between(post_t.week, counterfactual.values, post_t.bookings_indexed.values,
                alpha=0.15, color=BRAND_BLUE, label=f'Incremental lift (~{est_lift:.0%})')
ax.axvline(PRE_WEEKS + 0.5, color=GRAY, linestyle='--', lw=1)
ax.set_title('DiD: Observed vs. Counterfactual', fontsize=12, fontweight='bold')
ax.set_xlabel('Week')
ax.set_ylabel('Indexed Bookings')
ax.legend(fontsize=8)

# ── Right: Coefficient plot ────────────────────────────────────
ax2 = axes[1]
params = model.params.drop('Intercept')
ci     = model.conf_int().drop('Intercept')
labels = ['Treatment\n(baseline diff)', 'Post Period\n(time trend)', 'Treatment × Post\n(CAUSAL EFFECT)']
colors = [GRAY, GRAY, BRAND_BLUE]
y_pos  = range(len(params))

for i, (param, label, color) in enumerate(zip(params.index, labels, colors)):
    ax2.barh(i, params[param], color=color, alpha=0.7, height=0.5)
    ax2.errorbar(params[param], i,
                 xerr=[[params[param] - ci.loc[param, 0]], [ci.loc[param, 1] - params[param]]],
                 fmt='none', color='black', capsize=4, lw=1.5)

ax2.set_yticks(list(y_pos))
ax2.set_yticklabels(labels, fontsize=9)
ax2.axvline(0, color='black', lw=0.8)
ax2.set_title('Model Coefficients (95% CI)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Effect on Avg Weekly Bookings')

# Annotate the key coefficient
ax2.annotate(f'  β₃ = {did_coef:.0f}\n  p < 0.001',
             xy=(did_coef, 2), fontsize=9, color=BRAND_BLUE, fontweight='bold')

plt.tight_layout()
plt.savefig('did_results.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Section 6: Business Recommendation

This is where measurement becomes strategy.

A statistically significant result is necessary but not sufficient. The business needs to know: **so what do we do with the $15M?**

In [ ]:
# Business translation inputs
TOTAL_BUDGET_USD     = 15_000_000
AVG_BOOKING_VALUE    = 650          # average booking revenue ($)
TOTAL_TEST_BOOKINGS  = df[(df.is_treatment == 1) & (df.is_test == 1)]['bookings'].sum()

# Incremental bookings estimate
# DiD lift applied to total attributed bookings
total_attributed     = TOTAL_TEST_BOOKINGS
incremental_bookings = total_attributed * est_lift
incremental_revenue  = incremental_bookings * AVG_BOOKING_VALUE

# Scale to full Q3 ($15M budget, not just test period)
# Test was 8 weeks; Q3 = 13 weeks
scale_factor         = 13 / TEST_WEEKS
q3_incremental_rev   = incremental_revenue * scale_factor * (N_MARKETS / N_TREATMENT)
q3_iroas             = q3_incremental_rev / TOTAL_BUDGET_USD

# Scenario ranges (CI-based)
lift_low  = did_ci[0] / baseline
lift_high = did_ci[1] / baseline
iroas_low  = (total_attributed * lift_low  * AVG_BOOKING_VALUE * scale_factor * (N_MARKETS/N_TREATMENT)) / TOTAL_BUDGET_USD
iroas_high = (total_attributed * lift_high * AVG_BOOKING_VALUE * scale_factor * (N_MARKETS/N_TREATMENT)) / TOTAL_BUDGET_USD

print('=' * 60)
print('  Q3 PAID SEARCH — BUSINESS RECOMMENDATION')
print('=' * 60)
print(f'\n  Estimated incremental lift:    {est_lift:.1%}  (95% CI: {lift_low:.1%}–{lift_high:.1%})')
print(f'  Projected incremental revenue: ${q3_incremental_rev:>12,.0f}')
print(f'  Budget:                        ${TOTAL_BUDGET_USD:>12,.0f}')
print(f'  Incremental ROAS (iROAS):      {q3_iroas:.2f}x')
print(f'  iROAS range (conservative–optimistic): {iroas_low:.2f}x – {iroas_high:.2f}x')
print()
print('  RECOMMENDATION:')
if q3_iroas >= 1.5:
    print('  ✅ SCALE — iROAS exceeds 1.5x hurdle rate.')
    print('     Recommend full $15M deployment with quarterly refresh.')
elif q3_iroas >= 1.0:
    print('  ⚠️  HOLD — iROAS positive but below scale threshold.')
    print('     Recommend targeted deployment in top-performing markets.')
else:
    print('  ❌ REALLOCATE — iROAS below breakeven.')
    print('     Recommend shifting budget to higher-incrementality channels.')
print()
print('  KEY ASSUMPTIONS & LIMITATIONS:')
print(f'  • Average booking value assumed: ${AVG_BOOKING_VALUE} (sensitivity test recommended)')
print('  • Spillover between adjacent markets not modeled (likely conservative bias)')
print('  • Seasonality scaling from 8-week test to 13-week Q3 adds uncertainty')
print('  • Recommend re-test in Q4 to validate lift stability across seasons')
print('=' * 60)

---

## Section 7: What Would Change This Recommendation

A measurement framework isn't complete without explicitly stating its failure modes. This is the section that separates analysts from marketing scientists.

| Condition | Impact on Recommendation |
|---|---|
| Average booking value < $400 | iROAS falls below 1.0x — triggers reallocation |
| Significant spillover between test/control markets | Lift is understated — actual iROAS likely higher |
| Competitor spend increased in holdout markets | Lift is overstated — holdout wasn't a clean control |
| Seasonal pattern differs materially from Q3 baseline | Scale factor unreliable — run a Q4 validation test |
| Brand keyword vs. non-brand breakdown skewed | Incremental logic differs by keyword type — decompose before scaling |

---

## Summary

This notebook demonstrates a complete geo holdout framework for measuring paid search incrementality:

1. **Methodology selection** — geo holdout chosen over MMM/user A/B based on operational feasibility and causal validity for search
2. **Design rigor** — parallel trends validated pre-test; market selection criteria documented
3. **Causal estimation** — DiD model recovers the true treatment effect with statistical precision
4. **Business translation** — statistical output converted to iROAS and a concrete budget recommendation
5. **Intellectual honesty** — assumptions stated, failure modes enumerated, next steps defined

**The goal of incrementality measurement is not to produce a number. It's to give the business the confidence to act — or the clarity to stop.**

---
*Built with Python · pandas · statsmodels · matplotlib*  
*Framework applicable to travel, retail, and subscription e-commerce*